In [ ]:
!pip install requests pandas tqdm

In [1]:
import requests
import pandas as pd
import time
import logging
import datetime
from tqdm import tqdm

API_KEY = "93b69ce83cd74d7c07c885d29b0a24e9"

In [ ]:
YEARS = ["2026","2025","2024", "2023","2022","2021","2020","2019","2018","2017","2016","2015","2014", "2013","2012","2011","2010","2009","2008","2007","2006","2005","2004","2003","2002", "2001","2000"]

movie_ids = []

for year in YEARS:
    print("год",year)
    count_before = len(movie_ids)

    for page in range(1, 26):
        url = url = f"https://api.themoviedb.org/3/discover/movie?api_key={API_KEY}&page={page}&sort_by=popularity.desc&language=en-US&primary_release_year={year}"
        data = requests.get(url).json()
        results = data.get("results", [])
        movie_ids.extend([movie["id"] for movie in results])
        time.sleep(0.001)

    print(year,"--",len(movie_ids) - count_before,"      всего фильмов --", len(movie_ids))

print("Итого", len(movie_ids))

In [ ]:
movies_list = []

for movie_id in movie_ids:
    try:
        movie_en = requests.get(f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={API_KEY}&language=en-US").json()
        movie_ru = requests.get(f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={API_KEY}&language=ru-RU").json()
        credits = requests.get(f"https://api.themoviedb.org/3/movie/{movie_id}/credits?api_key={API_KEY}").json()
        keywords_data = requests.get(f"https://api.themoviedb.org/3/movie/{movie_id}/keywords?api_key={API_KEY}").json()

        cast = ", ".join([i["name"] for i in credits.get("cast", [])[:5]])
        director = next((a["name"] for a in credits.get("crew", []) if a["job"] == "Director"), None)
        keywords = ", ".join([k["name"] for k in keywords_data.get("keywords", [])[:5]])

        movies_list.append({
            "id": movie_en.get("id"),
            "title_en": movie_en.get("title"),
            "title_ru": movie_ru.get("title"),
            "overview": movie_en.get("overview"),
            "release_date": movie_en.get("release_date"),
            "genres": ", ".join([g["name"] for g in movie_en.get("genres", [])]),
            "runtime": movie_en.get("runtime"),
            "budget": movie_en.get("budget"),
            "revenue": movie_en.get("revenue"),
            "vote_average": movie_en.get("vote_average"),
            "vote_count": movie_en.get("vote_count"),
            "popularity": movie_en.get("popularity"),
            "original_language": movie_en.get("original_language"),
            "production_countries": ", ".join([c["name"] for c in movie_en.get("production_countries", [])]),
            "imdb_id": movie_en.get("imdb_id"),
            "cast": cast,
            "director": director,
            "keywords": keywords,
        })

        if len(movies_list) % 100 == 0:
            print("фильмов", len(movies_list),"/", len(movie_ids))

    except:
        print("не взятый фильм", movie_id)
        continue

    time.sleep(0.005)

print("Всего", len(movies_list))

фильмов 100 / 2640
не взятый фильм 1445363
фильмов 200 / 2640
не взятый фильм 1007757
не взятый фильм 1566589
не взятый фильм 1268609


In [ ]:
df_tmdb = pd.DataFrame(movies_list)
print(df_tmdb.shape)
df_tmdb.head(3)

In [ ]:
df_tmdb.to_csv("tmdb_movies.csv", index=False)